# Figure S1: Extended conditioning analysis

5 rows (A-E) x 3 property-pair columns:
- **A**: Experimental MIC landscape
- **B**: Experimental aggregation propensity
- **C**: Conditioning accuracy
- **D**: Predicted APEX activity
- **E**: Predicted aggregation propensity

In [ ]:
import pandas as pd, numpy as np, re, warnings
from pathlib import Path
from Bio import SeqIO
from scipy.stats import binned_statistic_2d, gaussian_kde
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt, matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from matplotlib.colors import TwoSlopeNorm
from mpl_toolkits.axes_grid1 import make_axes_locatable
import modlamp.analysis as manalysis
warnings.filterwarnings('ignore')

plt.style.use('default')
plt.rcParams.update({
    'font.size':7,'axes.titlesize':7,'axes.labelsize':6,
    'xtick.labelsize':6,'ytick.labelsize':6,'legend.fontsize':6,
    'axes.linewidth':0.5,'axes.edgecolor':'grey',
    'xtick.color':'#555555','ytick.color':'#555555',
    'axes.labelcolor':'#333333','font.family':'sans-serif',
    'font.sans-serif':['Arial','DejaVu Sans'],
    'figure.dpi':300,'savefig.dpi':300,
    'xtick.major.width':0.5,'ytick.major.width':0.5,
    'xtick.major.size':2,'ytick.major.size':2,
    'axes.spines.top':True,'axes.spines.right':True,
    'figure.facecolor':'white','axes.facecolor':'white',
})

ROOT=Path("../data")
SPECIES=ROOT/"figure2/species"; TRAINING=ROOT/"figure2/training/generative-model-dataset.csv"
TARGETED=ROOT/"figure2/conditioning"; APEX_DIR=ROOT/"figure2/apex"; SWEEP=ROOT/"figure3/sweep"
CHARGE_GRID=[0,2,4,6,8,10,12]; HYDRO_GRID=[-0.75,-0.50,-0.25,0.0,0.25,0.50,0.75]; LENGTH_GRID=[5,10,15,20,25,30,35]

AGGRESCAN={'I':1.822,'F':1.754,'V':1.594,'L':1.38,'Y':1.159,'W':1.037,'M':0.91,'C':0.604,
           'A':-0.036,'T':-0.159,'S':-0.294,'P':-0.334,'G':-0.535,'K':-0.931,'H':-1.033,
           'Q':-1.231,'R':-1.24,'N':-1.302,'E':-1.412,'D':-1.836}
AA_MW={'A':89.09,'R':174.20,'N':132.12,'D':133.10,'C':121.16,'E':147.13,'Q':146.15,
       'G':75.03,'H':155.16,'I':131.17,'L':131.17,'K':146.19,'M':149.21,'F':165.19,
       'P':115.13,'S':105.09,'T':119.12,'W':204.23,'Y':181.19,'V':117.15}
def peptide_mw(seq): return sum(AA_MW.get(aa,0) for aa in str(seq).upper())-(len(str(seq))-1)*18.015
def bcharge(seqs): h=manalysis.GlobalAnalysis(list(seqs)); h.calc_charge(); return h.charge[0]
def bhydro(seqs): h=manalysis.GlobalAnalysis(list(seqs)); h.calc_H(scale='eisenberg'); return h.H[0]
def seq_agg(s):
    sc=[AGGRESCAN.get(aa,0) for aa in str(s).upper() if aa in AGGRESCAN]
    if len(sc)<5: return np.mean(sc) if sc else np.nan
    return np.mean([np.mean(sc[i:i+5]) for i in range(len(sc)-4)])
def calc_props(seqs):
    s=list(seqs); return pd.DataFrame({'charge':bcharge(s),'length':[len(x) for x in s],'hydrophobicity':bhydro(s)})
def style_spines(ax):
    for sp in ax.spines.values(): sp.set_edgecolor('grey'); sp.set_linewidth(0.5)
def parse_hydro(s):
    if s.startswith('hm'): return -float(s[2:].replace('p','.'))
    elif s.startswith('h'): return float(s[1:].replace('p','.'))
    return None
def parse_dir(dn,pt):
    parts=dn.split('_')
    if pt=='ch': return int(parts[0][1:]),parse_hydro(parts[1])
    elif pt=='cl': return int(parts[0][1:]),int(parts[1][1:])
    elif pt=='hl': return parse_hydro(parts[0]),int(parts[1][1:])
    return None,None
def is_allowed(xt,yt,pt):
    tol=0.01
    if pt=='ch': return xt in CHARGE_GRID and any(abs(yt-h)<tol for h in HYDRO_GRID)
    elif pt=='cl': return xt in CHARGE_GRID and yt in LENGTH_GRID
    elif pt=='hl': return any(abs(xt-h)<tol for h in HYDRO_GRID) and yt in LENGTH_GRID
    return False

# ================================================================
print("Loading data...")
sp_dfs=[pd.read_csv(f) for f in SPECIES.glob("*.csv")]
sp_df=pd.concat(sp_dfs,ignore_index=True)
sp_agg=sp_df.groupby('sequence').agg(activity=('activity','mean')).reset_index()
sp_agg['mw']=sp_agg['sequence'].apply(peptide_mw)
sp_agg['activity_um']=sp_agg['activity']/(sp_agg['mw']/1000)
sp_agg['aggregation']=sp_agg['sequence'].apply(seq_agg)
sp_agg['charge']=bcharge(sp_agg['sequence'].tolist())
sp_agg['hydrophobicity']=bhydro(sp_agg['sequence'].tolist())
sp_agg['length']=sp_agg['sequence'].apply(len)
training=pd.read_csv(TRAINING)
amp_props=calc_props(training[training['IsAMP']==1]['Sequence'].tolist())

GRID_DEFS=[
    ('grid-sweep-charge-hydrophob','ch','charge','hydrophobicity','Charge','Hydrophobicity'),
    ('grid-sweep-charge-length','cl','charge','length','Charge','Length'),
    ('grid-sweep-hydrophob-length','hl','hydrophobicity','length','Hydrophobicity','Length'),
]
all_grid={}
for pi,(gd,pt,xp,yp,xl,yl) in enumerate(GRID_DEFS):
    cb=TARGETED/gd; ab=APEX_DIR/gd; rows=[]; seen=set()
    if not cb.exists(): all_grid[pi]=pd.DataFrame(); continue
    for sd in sorted(cb.iterdir()):
        if not sd.is_dir(): continue
        fp=sd/"samples.fasta"
        if not fp.exists(): continue
        xt,yt=parse_dir(sd.name,pt)
        if xt is None or not is_allowed(xt,yt,pt): continue
        key=(round(xt,2),round(yt,2))
        if key in seen: continue
        seen.add(key)
        seqs=[str(r.seq) for r in SeqIO.parse(fp,'fasta')]
        if not seqs: continue
        props=calc_props(seqs)
        if xp=='charge': ok_x=np.abs(props['charge']-xt)<=0.25
        elif xp=='hydrophobicity': ok_x=np.abs(props['hydrophobicity']-xt)<=0.1
        else: ok_x=props[xp]==int(xt)
        if yp=='hydrophobicity': ok_y=np.abs(props['hydrophobicity']-yt)<=0.1
        elif yp=='length': ok_y=props['length']==int(yt)
        else: ok_y=np.abs(props['charge']-yt)<=0.25
        apex_mic=np.nan
        af=ab/f"{sd.name}-apex-predictions-min.tsv"
        if af.exists(): adf=pd.read_csv(af,sep='\t'); apex_mic=np.log2(adf.groupby('Sequence_id')['MIC'].min().mean())
        rows.append({'xt':xt,'yt':yt,'x_actual':props[xp].mean(),'y_actual':props[yp].mean(),
                     'success':(ok_x&ok_y).mean(),'apex':apex_mic,'agg':np.mean([seq_agg(s) for s in seqs])})
    all_grid[pi]=pd.DataFrame(rows)
    print(f"  {xl} vs {yl}: {len(rows)} pts")

In [ ]:
# ================================================================
# FIGURE S1: CONDITIONING  6.5 x 8
# Panel labels placed via fig.text at fixed x, outside axes
# ================================================================
print("\n=== Figure S1 ===")
fig=plt.figure(figsize=(6.5,8.0),dpi=300)
PL=dict(fontsize=10,fontweight='bold',va='top',ha='left')

gs=gridspec.GridSpec(5,3,figure=fig,
    left=0.10,right=0.94,top=0.98,bottom=0.03,
    wspace=0.42, hspace=0.32,
    height_ratios=[1,1,1,1,1])

pxl=['Charge','Charge','Hydrophobicity']
pyl=['Hydrophobicity','Length','Length']
pxc=['charge','charge','hydrophobicity']
pyc=['hydrophobicity','length','length']
pranges=[[[0,30],[-2.5,1.0]],[[0,30],[0,80]],[[-2,1],[0,80]]]
gxl=[(-2.5,13.5),(-2.5,13.5),(-1,1)]
gyl=[(-1,1),(0,38),(0,38)]
amp_pc=[('charge','hydrophobicity'),('charge','length'),('hydrophobicity','length')]

# Use fig.text for panel labels at a fixed x=0.01 to avoid axis overlap
row_map = {0:'A', 1:'B', 2:'C', 3:'D', 4:'E'}

# Row 0=A, Row 1=B
for ri in [0,1]:
    for ci in range(3):
        ax=fig.add_subplot(gs[ri,ci])
        if ci==0:
            # Place label at fixed x in figure coords, y from axis top
            bbox=ax.get_position()
            fig.text(0.01, bbox.y1, row_map[ri], **PL)
        x=sp_agg[pxc[ci]].values; y=sp_agg[pyc[ci]].values
        if ri==0:
            z=np.log2(sp_agg['activity_um'].clip(1)).values
            stat,xe,ye,_=binned_statistic_2d(x,y,z,statistic='mean',bins=[25,20],range=pranges[ci])
            im=ax.imshow(stat.T,origin='lower',aspect='auto',cmap='RdBu',norm=TwoSlopeNorm(vcenter=5,vmin=0,vmax=7),
                         extent=[xe[0],xe[-1],ye[0],ye[-1]],interpolation='nearest')
            cbl_text=r'log$_2$ MIC ($\mu$M)'
        else:
            z=sp_agg['aggregation'].values
            stat,xe,ye,_=binned_statistic_2d(x,y,z,statistic='mean',bins=[25,20],range=pranges[ci])
            im=ax.imshow(stat.T,origin='lower',aspect='auto',cmap='RdYlBu_r',norm=TwoSlopeNorm(vcenter=0,vmin=-0.5,vmax=1.0),
                         extent=[xe[0],xe[-1],ye[0],ye[-1]],interpolation='nearest')
            cbl_text='Aggregation Propensity'
        ax.set_xlabel(pxl[ci],labelpad=1,fontsize=6); ax.set_ylabel(pyl[ci],labelpad=1,fontsize=6)
        ax.tick_params(axis='both',length=2,width=0.5,labelsize=6); style_spines(ax)
        if ci==2:
            d=make_axes_locatable(ax); cx=d.append_axes("right",size="5%",pad=0.04)
            cb=plt.colorbar(im,cax=cx); cb.set_label(cbl_text,fontsize=6); cb.ax.tick_params(labelsize=6)

# Rows 2,3,4 = C,D,E
row_cfgs=[
    (2,'C','success','RdYlGn',0,1,'Conditioning Accuracy (%)','darkred',None),
    (3,'D','apex','RdBu',0,7,r'log$_2$ $MIC_{APEX}$ ($\mu$M)','orange',5),
    (4,'E','agg','RdYlBu_r',-0.5,1.5,'Aggregation Propensity','darkgreen',0),
]
for ri,pl,ck,cm,vn,vx,cbl,xmc,vc in row_cfgs:
    for ci in range(3):
        ax=fig.add_subplot(gs[ri,ci])
        if ci==0:
            bbox=ax.get_position()
            fig.text(0.01, bbox.y1, pl, **PL)
        gdf=all_grid[ci]
        if len(gdf)==0: continue
        ac2,ah2=amp_pc[ci]; acv=amp_props[ac2].values; ahv=amp_props[ah2].values
        v=~np.isnan(acv)&~np.isnan(ahv)
        try:
            kde=gaussian_kde(np.vstack([acv[v],ahv[v]]),bw_method='scott')
            cg=np.linspace(acv[v].min()-1,acv[v].max()+1,100)
            hg=np.linspace(ahv[v].min()-0.2,ahv[v].max()+0.2,100)
            Cg,Hg=np.meshgrid(cg,hg)
            Zg=kde(np.vstack([Cg.ravel(),Hg.ravel()])).reshape(Cg.shape)
            ax.contourf(Cg,Hg,Zg,levels=10,cmap='Greys',alpha=0.3)
            ax.contour(Cg,Hg,Zg,levels=5,colors='gray',linewidths=0.4,alpha=0.5,linestyles='dashed')
        except: pass
        snorm=TwoSlopeNorm(vcenter=vc,vmin=vn,vmax=vx) if vc is not None else plt.Normalize(vmin=vn,vmax=vx)
        vr=gdf[gdf[ck].notna()]
        if len(vr)>0:
            ax.scatter(vr['x_actual'],vr['y_actual'],c=vr[ck],s=16,cmap=cm,norm=snorm,
                       edgecolors='black',linewidths=0.25,alpha=0.9,zorder=10)
        ax.scatter(gdf['xt'],gdf['yt'],s=18,marker='x',c=xmc,linewidths=0.5,alpha=0.7,zorder=5)
        ax.set_xlabel(pxl[ci],fontsize=6,labelpad=1)
        if ci==0: ax.set_ylabel(pyl[ci],fontsize=6,labelpad=1)
        ax.set_xlim(gxl[ci]); ax.set_ylim(gyl[ci])
        ax.grid(True,alpha=0.2,linestyle='--',linewidth=0.25)
        ax.tick_params(labelsize=6,length=2); style_spines(ax)
        if ci==2:
            sm=plt.cm.ScalarMappable(cmap=cm,norm=snorm)
            d=make_axes_locatable(ax); cx=d.append_axes("right",size="5%",pad=0.04)
            cb=plt.colorbar(sm,cax=cx); cb.set_label(cbl,fontsize=6); cb.ax.tick_params(labelsize=6)

out1='../figures/figureS1_conditioning.png'
plt.savefig(out1,bbox_inches='tight',dpi=300)
plt.savefig(out1.replace('.png','.pdf'),bbox_inches='tight')
print(f"Saved {out1}")
plt.close()